# ETL Pipeline: Digital Desert in NE Cambodia
## Bridging the Digital Divide Hackathon

This notebook extracts, transforms, and loads data from multiple sources to identify connectivity gaps in North-eastern Cambodia, particularly affecting indigenous communities.

## 1. Setup & Dependencies

In [1]:
import pandas as pd
import geopandas as gpd
import json
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('./archive')
print('Environment ready for ETL processing')

Environment ready for ETL processing


## 2. Define Target Regions & Provinces

Focus areas: Ratanakiri, Mondulkiri, Kratie, Stung Treng, Kampong Thom, Preah Vihear

In [2]:
TARGET_PROVINCES = [
    'Ratanakiri',
    'Mondulkiri',
    'Kratie',
    'Stung Treng',
    'Kampong Thom',
    'Preah Vihear'
]

CRS_UTM48N = 'EPSG:32648'
CRS_WGS84 = 'EPSG:4326'

print(f'Target provinces: {TARGET_PROVINCES}')
print(f'Primary CRS: {CRS_UTM48N} (UTM Zone 48N for Cambodia)')

Target provinces: ['Ratanakiri', 'Mondulkiri', 'Kratie', 'Stung Treng', 'Kampong Thom', 'Preah Vihear']
Primary CRS: EPSG:32648 (UTM Zone 48N for Cambodia)


## 3. Extract: Load All Raw Data Sources

In [3]:
# Load World Bank Findex data (connectivity & financial inclusion)
findex = pd.read_csv(DATA_DIR / 'Findex_Microdata_2025_updateCambodia.csv')
print(f'Findex data shape: {findex.shape}')
print(f'Columns: {findex.columns.tolist()[:10]}...')
findex.head()

Findex data shape: (1000, 199)
Columns: ['year', 'economy', 'economycode', 'regionwb', 'pop_adult', 'wpid_random', 'wgt', 'female', 'age', 'educ']...


,year,economy,economycode,regionwb,pop_adult,wpid_random,wgt,female,age,educ,...,fin48e,fin48f,fin49a,fin49b,fin49c,fin49d,fin49e,fin49f,fin50,fin51
0,2024,Cambodia,KHM,East Asia & Pacific (excluding high income),12177453,111158734,0.669096,1,53,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024,Cambodia,KHM,East Asia & Pacific (excluding high income),12177453,111257513,0.203102,1,42,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024,Cambodia,KHM,East Asia & Pacific (excluding high income),12177453,111274044,3.160188,2,31,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024,Cambodia,KHM,East Asia & Pacific (excluding high income),12177453,111354708,0.668263,1,22,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024,Cambodia,KHM,East Asia & Pacific (excluding high income),12177453,111362717,0.203102,1,55,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Load FAO flood event data
floods = pd.read_csv(DATA_DIR / 'khm-flood-events-fao-eve.csv')
print(f'Flood events shape: {floods.shape}')
print(f'Columns: {floods.columns.tolist()}')
floods.head()

Flood events shape: (6532, 17)
Columns: ['adm0_iso3', 'adm0_name', 'admin_level', 'adm1_pcode', 'adm1_name', 'adm2_pcode', 'adm2_name', 'period_number', 'start_date', 'end_date', 'cropland_flooded_sq_km', 'cropland_flooded_ha', 'total_area_flooded_sq_km', 'total_area_flooded_ha', 'perc_cropland_flooded', 'perc_total_area_flooded', 'pop_exposed']


,adm0_iso3,adm0_name,admin_level,adm1_pcode,adm1_name,adm2_pcode,adm2_name,period_number,start_date,end_date,cropland_flooded_sq_km,cropland_flooded_ha,total_area_flooded_sq_km,total_area_flooded_ha,perc_cropland_flooded,perc_total_area_flooded,pop_exposed
0,KHM,Cambodia,admin2,KH06,Kampong Thom,KH0601,Baray,53,2026-03-01,2026-03-15,28.02,2802,61.55,6155,3.045041,4.559431,3566
1,KHM,Cambodia,admin2,KH05,Kampong Speu,KH0508,Thpong,53,2026-03-01,2026-03-15,0.61,61,2.27,227,0.128707,0.311290,557
2,KHM,Cambodia,admin2,KH05,Kampong Speu,KH0507,Samraong Tong,53,2026-03-01,2026-03-15,11.44,1144,12.48,1248,2.485628,1.655111,2338
3,KHM,Cambodia,admin2,KH05,Kampong Speu,KH0506,Phnum Sruoch,53,2026-03-01,2026-03-15,0.07,7,0.68,68,0.010282,0.040762,18
4,KHM,Cambodia,admin2,KH05,Kampong Speu,KH0505,Odongk,53,2026-03-01,2026-03-15,11.95,1195,13.19,1319,3.301284,2.459848,3355


In [5]:
# Load indigenous communities by MRD (Ministry of Rural Development)
with open(DATA_DIR / 'indigenous_communities_by_mrd.json', 'r', encoding='utf-8') as f:
    ip_mrd_geom = json.load(f)

ip_mrd = gpd.GeoDataFrame.from_features(
    ip_mrd_geom['features'],
    crs=CRS_UTM48N
)
print(f'Indigenous communities (MRD) shape: {ip_mrd.shape}')
print(f'Geometry type: {ip_mrd.geometry.type.unique()}')
print(f'Provinces: {ip_mrd["province"].unique()}')
ip_mrd.head()

Indigenous communities (MRD) shape: (58, 12)
Geometry type: ['MultiPoint']
Provinces: ['Ratanakiri']


,geometry,map_id,ethnic,village,commune,district,province,requested_date_by_community,identified_date_by_provincial_govenor,identified_date_by_mrd,identified_by_mrd,reference
0,MULTIPOINT (753135.511 1516543.261),1,Jarai,Le,Lum Choar,Ou Ya Dav,Ratanakiri,05/06/2010,05/27/2010,09/23/2010,Yes,Letter_from_MRD_Lae_village_Ratanakiri__23 _09...
1,MULTIPOINT (724489.697 1532509.079),2,Kreung,Kralong,L'ak,Ou Chum,Ratanakiri,11/03/2011,12/30/2011,07/19/2012,Yes,Letter_from_MRD_Lae_village_Ratanakiri__23 _09...
2,MULTIPOINT (741067.605 1515584.243),3,Jarai,Sala,Kak,Bar Kaev,Ratanakiri,03/04/2010,04/23/2010,05/24/2010,Yes,Letter_from_MRD_Lae_village_Ratanakiri__23 _09...
3,MULTIPOINT (747949.578 1503741.348),4,Jarai,Ta Kok Chor Ray,Bar Kham,Ou Ya Dav,Ratanakiri,12/06/2011,12/30/2011,07/19/2012,Yes,Letter_from_MRD_Lae_village_Ratanakiri__23 _09...
4,MULTIPOINT (700969.915 1512923.183),5,Kreung,Teum,Teun,Koun Mom,Ratanakiri,04/29/2013,06/06/2013,09/02/2013,Yes,Letter_from_MRD_Lae_village_Ratanakiri__23 _09...


In [6]:
# Load registered indigenous communal lands
with open(DATA_DIR / 'regis_ip_english4.json', 'r', encoding='utf-8') as f:
    ip_reg_geom = json.load(f)

ip_reg = gpd.GeoDataFrame.from_features(
    ip_reg_geom['features'],
    crs=CRS_WGS84
)
ip_reg = ip_reg.to_crs(CRS_UTM48N)
print(f'Registered indigenous lands shape: {ip_reg.shape}')
print(f'Provinces: {ip_reg["province"].unique()}')
ip_reg.head()

Registered indigenous lands shape: (44, 19)
Provinces: ['Mondulkiri' 'Kratie' 'Ratanakiri' 'Stung Treng']


,geometry,map_id,ip_name,num_family,village,commune,district,province,land_size,titled_par,residentia,tradi_agri,swidden_fa,burial _fo,forest_lan,date_titli,reference,latest_upd,language
0,MULTIPOINT (737679.919 1375715.254),ip_01,Phnnong,116,Pu Trom,Romonea,Sen Monorom,Mondulkiri,1005,53,24,29,Not found,Not found,Not found,Not found,Sub_decree_No_128__16.09.2015.pdf,15/2/2016,English
1,MULTIPOINT (619075.692 1445133.575),ip_08,Phnnong,37,Ou Kak,Ou Krieng,Sambour,Kratie,402.04,21,5,9,6,Not found,1,Not found,Sub_decree_No_186__24.12.2015.pdf,15/2/2016,English
2,MULTIPOINT (700645.217 1353833.331),ip_02,Phnnong,27,Ou Chrar,Srae Preah,Keo Seima,Mondulkiri,293.51,24,10,14,Not found,Not found,Not found,Not found,Sub_decree_No_62__13.02.2013.pdf,15/2/2016,English
3,MULTIPOINT (701307.203 1358636.300),ip_03,Phnnong,45,Hkati,Srae Preah,Keo Seima,Mondulkiri,417.8,18,8,10,Not found,Not found,Not found,07/03/2013,Hkati_land_title__07.03.2013.pdf;Sub_decree_No...,15/2/2016,English
4,MULTIPOINT (703877.207 1347277.382),ip_04,Phnnong,31,Srae Lvi,Srae Khtum,Keo Seima,Mondulkiri,358.51,Not found,Not found,Not found,Not found,Not found,Not found,Not found,Sub_decree_No_239__16.05.2013.pdf,15/2/2016,English


In [7]:
# Load HOT OSM buildings (may be large)
try:
    buildings = gpd.read_file(DATA_DIR / 'hotosm_khm_buildings_polygons_geojson.geojson')
    print(f'Buildings shape: {buildings.shape}')
    print(f'CRS: {buildings.crs}')
    buildings.head()
except Exception as e:
    print(f'Note: Buildings file may be large or require special handling: {e}')

Note: Buildings file may be large or require special handling: module 'fiona' has no attribute 'path'


## 4. Transform: Clean & Filter Data

In [ ]:
# Load cell tower locations (GSM towers)
cell_towers = pd.read_csv(
    DATA_DIR / '452.csv',
    header=None,
    names=['network_type', 'country_code', 'mcc', 'mnc', 'lac', 'cid', 'lon', 'lat', 'range', 'samples', 'changeable', 'created', 'updated', 'avg_distance']
)

# Convert to GeoDataFrame with proper CRS (WGS84)
from shapely.geometry import Point

cell_towers_gdf = gpd.GeoDataFrame(
    cell_towers.drop(['lon', 'lat'], axis=1),
    geometry=[Point(xy) for xy in zip(cell_towers['lon'], cell_towers['lat'])],
    crs=CRS_WGS84
)
cell_towers_gdf = cell_towers_gdf.to_crs(CRS_UTM48N)

print(f'Cell tower records: {cell_towers_gdf.shape[0]}')
print(f'CRS: {cell_towers_gdf.crs}')
print(f'Bounds (UTM48N): {cell_towers_gdf.total_bounds}')
cell_towers_gdf.head()


In [8]:
# Filter floods to target provinces
floods_filtered = floods[floods['adm1_name'].isin(TARGET_PROVINCES)].copy()
print(f'Filtered flood events: {floods_filtered.shape[0]} records')
print(f'Provinces with flood data: {floods_filtered["adm1_name"].unique().tolist()}')
floods_filtered.head()

Filtered flood events: 975 records
Provinces with flood data: ['Kampong Thom', 'Preah Vihear', 'Kratie', 'Stung Treng']


,adm0_iso3,adm0_name,admin_level,adm1_pcode,adm1_name,adm2_pcode,adm2_name,period_number,start_date,end_date,cropland_flooded_sq_km,cropland_flooded_ha,total_area_flooded_sq_km,total_area_flooded_ha,perc_cropland_flooded,perc_total_area_flooded,pop_exposed
0,KHM,Cambodia,admin2,KH06,Kampong Thom,KH0601,Baray,53,2026-03-01,2026-03-15,28.02,2802,61.55,6155,3.045041,4.559431,3566
44,KHM,Cambodia,admin2,KH06,Kampong Thom,KH0602,Kampong Svay,53,2026-03-01,2026-03-15,7.50,750,63.35,6335,0.944290,4.396868,1398
45,KHM,Cambodia,admin2,KH06,Kampong Thom,KH0603,Stueng Saen,53,2026-03-01,2026-03-15,2.82,282,7.58,758,0.920563,2.008265,757
46,KHM,Cambodia,admin2,KH06,Kampong Thom,KH0605,Prasat Sambour,53,2026-03-01,2026-03-15,0.68,68,1.69,169,0.312155,0.223092,109
57,KHM,Cambodia,admin2,KH13,Preah Vihear,KH1308,Preah Vihear,53,2026-03-01,2026-03-15,0.79,79,9.73,973,1.253706,3.334236,603


## 4.5 Create Unified Dataset: Merge All Sources

Before cleaning, combine all data sources into one master dataset to enable cross-source analysis.

In [ ]:
# Create unified raw dataset combining all sources
# Start with all raw datasets without filtering

datasets = {
    'findex': {
        'data': findex,
        'type': 'attribute',
        'size': findex.shape[0],
        'description': 'World Bank Findex - connectivity & financial inclusion'
    },
    'floods': {
        'data': floods,
        'type': 'attribute', 
        'size': floods.shape[0],
        'description': 'FAO EVE - flood events'
    },
    'ip_communities_mrd': {
        'data': ip_mrd,
        'type': 'geospatial',
        'size': ip_mrd.shape[0],
        'description': 'Indigenous communities by Ministry of Rural Development'
    },
    'ip_registered_lands': {
        'data': ip_reg,
        'type': 'geospatial',
        'size': ip_reg.shape[0],
        'description': 'Registered indigenous communal lands'
    },
    'cell_towers': {
        'data': cell_towers_gdf,
        'type': 'geospatial',
        'size': cell_towers_gdf.shape[0],
        'description': 'GSM cell tower locations'
    }
}

print('=' * 70)
print('UNIFIED DATASET INVENTORY')
print('=' * 70)
for name, info in datasets.items():
    print(f"{name:20} | Type: {info['type']:12} | Records: {info['size']:>7} | {info['description']}")
print('=' * 70)

total_records = sum(info['size'] for info in datasets.values())
print(f"\nTotal records across all sources: {total_records:,}")


In [ ]:
# Create a master unified geospatial dataset with spatial joins
# Start with indigenous communities as anchor (most relevant for digital divide)

unified_raw = ip_mrd.copy()
unified_raw['source_primary'] = 'mrd_communities'

print(f'Starting unified dataset: {unified_raw.shape}')
print(f'Columns: {unified_raw.columns.tolist()}')

# Buffer indigenous communities for proximity analysis
unified_raw['geometry_buffered'] = unified_raw.geometry.buffer(15000)  # 15km buffer for cell tower analysis

# Spatial join with cell towers (find towers within buffer)
unified_with_towers = gpd.sjoin(
    unified_raw,
    cell_towers_gdf[['geometry']],
    how='left',
    predicate='intersects'
)

print(f'\nAfter spatial join with cell towers:')
print(f'  Shape: {unified_with_towers.shape}')
print(f'  Communities with cell towers: {unified_with_towers.groupby(level=0).size().sum()}')

# Spatial join with registered lands
unified_with_lands = gpd.sjoin(
    unified_raw[['geometry']],
    ip_reg[[col for col in ip_reg.columns if col != 'geometry'] + ['geometry']],
    how='left', 
    predicate='intersects'
)

print(f'\nAfter spatial join with registered lands:')
print(f'  Communities with registered lands: {(~unified_with_lands.index.isna()).sum()}')

# Add flood context from province-level data
unified_raw = unified_raw.merge(
    floods.groupby('adm1_name').agg({
        'pop_exposed': 'sum',
        'total_area_flooded_sq_km': 'sum'
    }).reset_index().rename(columns={'adm1_name': 'province'}),
    on='province',
    how='left'
)

print(f'\nUnified dataset structure after merging:')
print(f'  Total records: {unified_raw.shape[0]}')
print(f'  Total columns: {unified_raw.shape[1]}')
print(f'  Columns: {unified_raw.columns.tolist()}')


## 5. Clean & Transform: Unified Dataset

Now clean and standardize the merged dataset for analysis.

In [ ]:
# Data quality assessment on unified dataset
print('DATA QUALITY ASSESSMENT')
print('=' * 70)

for col in unified_raw.columns:
    non_null = unified_raw[col].notna().sum()
    null_pct = (1 - non_null / len(unified_raw)) * 100
    dtype = unified_raw[col].dtype
    
    if null_pct > 0:
        print(f'{col:30} | {dtype:10} | {null_pct:5.1f}% missing')

print('\nMissing data will be handled in type-specific cleaning steps.')


In [ ]:
# Clean unified dataset - standardize types and handle nulls
unified_clean = unified_raw.copy()

# Ensure province is consistent (string, title case)
if 'province' in unified_clean.columns:
    unified_clean['province'] = unified_clean['province'].str.strip().str.title()

# Standardize numeric columns
numeric_cols = ['pop_exposed', 'total_area_flooded_sq_km']
for col in numeric_cols:
    if col in unified_clean.columns:
        unified_clean[col] = pd.to_numeric(unified_clean[col], errors='coerce')

# Extract coordinates for analysis
unified_clean['x'] = unified_clean.geometry.apply(
    lambda geom: geom.geoms[0].x if hasattr(geom, 'geoms') else geom.x
)
unified_clean['y'] = unified_clean.geometry.apply(
    lambda geom: geom.geoms[0].y if hasattr(geom, 'geoms') else geom.y
)

print(f'✓ Unified dataset cleaned: {unified_clean.shape[0]} records')
print(f'\nSummary statistics for key fields:')
print(unified_clean[['province', 'ethnic', 'pop_exposed', 'total_area_flooded_sq_km']].describe())


In [ ]:
# Filter unified dataset to target provinces
unified_final = unified_clean[unified_clean['province'].isin(TARGET_PROVINCES)].copy()

print(f'Unified dataset filtered to {len(TARGET_PROVINCES)} target provinces:')
print(f'  Final record count: {unified_final.shape[0]}')
print(f'  Provinces included: {sorted(unified_final["province"].unique())}')
print(f'\nDataset ready for analysis and export!')

# Save the unified raw dataset
unified_final.to_file(
    Path('./processed_data') / '00_unified_dataset_all_sources.geojson',
    driver='GeoJSON'
)
print('✓ Exported: unified_dataset_all_sources.geojson')


In [9]:
# Filter indigenous communities to target provinces
ip_mrd_filtered = ip_mrd[ip_mrd['province'].isin(TARGET_PROVINCES)].copy()
ip_reg_filtered = ip_reg[ip_reg['province'].isin(TARGET_PROVINCES)].copy()

print(f'Indigenous communities (MRD): {ip_mrd_filtered.shape[0]} villages')
print(f'Registered indigenous lands: {ip_reg_filtered.shape[0]} communal areas')
print(f'\nIndigenous groups (MRD):' )
print(ip_mrd_filtered['ethnic'].value_counts())

Indigenous communities (MRD): 58 villages
Registered indigenous lands: 44 communal areas

Indigenous groups (MRD):
ethnic
Jarai      17
Kreung     15
Tumpoun    15
Brao        8
Kachak      1
Kalaet      1
Kavet       1
Name: count, dtype: int64


In [13]:
# Create connectivity metrics from Findex
connectivity_cols = [col for col in findex.columns if col.startswith(('con', 'internet'))]
print(f'Connectivity columns: {connectivity_cols}')

# Aggregate connectivity statistics by demographic group
findex_connectivity = findex[[
    'age', 'educ', 'inc_q', 'emp_in', 'urbanicity', 'internet_use'
    ] + connectivity_cols].copy()

print(f'\nFindex connectivity data prepared: {findex_connectivity.shape}')
print(f'Data types:\n{findex_connectivity.dtypes.head()}')

Connectivity columns: ['internet_use', 'con1', 'con2a', 'con2b', 'con2c', 'con2d', 'con2e', 'con2f', 'con2g', 'con3', 'con4', 'con5', 'con6', 'con7', 'con8', 'con9', 'con10', 'con11', 'con12', 'con13', 'con14', 'con15', 'con16', 'con17', 'con18', 'con19', 'con20', 'con21', 'con22', 'con23', 'con24', 'con25', 'con26', 'con27', 'con28', 'con29', 'con30a', 'con30b', 'con30c', 'con30d', 'con30e', 'con30f', 'con30g', 'con30h', 'con31a', 'con31b', 'con31c', 'con31d', 'con31e', 'con31f', 'con31g', 'con31h', 'con32']

Findex connectivity data prepared: (1000, 59)
Data types:
age           int64
educ          int64
inc_q         int64
emp_in        int64
urbanicity    int64
dtype: object


## 6. Spatial Processing: Derived Features & Overlays

In [15]:
# Extract coordinates from MRD communities for distance/proximity analysis
ip_mrd_coords = ip_mrd_filtered.copy()

# Handle MultiPoint geometries - extract first point
ip_mrd_coords['x'] = ip_mrd_coords.geometry.apply(lambda geom: geom.geoms[0].x if hasattr(geom, 'geoms') else geom.x)
ip_mrd_coords['y'] = ip_mrd_coords.geometry.apply(lambda geom: geom.geoms[0].y if hasattr(geom, 'geoms') else geom.y)

print(f'Indigenous communities with coordinates: {ip_mrd_coords.shape[0]}')
print(f'Coordinate bounds (UTM48N):')
print(f'  X: {ip_mrd_coords["x"].min():.0f} to {ip_mrd_coords["x"].max():.0f}')
print(f'  Y: {ip_mrd_coords["y"].min():.0f} to {ip_mrd_coords["y"].max():.0f}')

Indigenous communities with coordinates: 58
Coordinate bounds (UTM48N):
  X: 699308 to 765045
  Y: 1472763 to 1557761


In [16]:
# Create a simple flood risk index for provinces
flood_risk = floods_filtered.groupby('adm1_name').agg({
    'pop_exposed': 'sum',
    'total_area_flooded_sq_km': 'sum',
    'adm2_name': 'nunique'
}).rename(columns={'adm2_name': 'affected_districts'}).reset_index()

flood_risk.columns = ['province', 'total_pop_exposed', 'total_area_flooded_km2', 'affected_districts']
flood_risk['risk_score'] = (flood_risk['total_pop_exposed'] / 1000) + (flood_risk['total_area_flooded_km2'])

print('Flood Risk by Province (NE Cambodia):')
print(flood_risk.sort_values('risk_score', ascending=False))

Flood Risk by Province (NE Cambodia):
       province  total_pop_exposed  total_area_flooded_km2  \
0  Kampong Thom            2889787                57019.51   
1        Kratie            1095896                13281.36   
3   Stung Treng             198183                 8734.43   
2  Preah Vihear             113348                 4205.27   

   affected_districts  risk_score  
0                   8   59909.297  
1                   6   14377.256  
3                   5    8932.613  
2                   8    4318.618  


## 7. Aggregate: Create Analysis-Ready Datasets

In [18]:
# Master indigenous communities dataset
ip_master = ip_mrd_filtered[[
    'map_id', 'ethnic', 'village', 'commune', 'district', 'province',
    'requested_date_by_community', 'identified_by_mrd', 'geometry'
]].copy()

ip_master['data_source'] = 'MRD_communities'
ip_master['x'] = ip_master.geometry.apply(lambda geom: geom.geoms[0].x if hasattr(geom, 'geoms') else geom.x)
ip_master['y'] = ip_master.geometry.apply(lambda geom: geom.geoms[0].y if hasattr(geom, 'geoms') else geom.y)

print(f'Master indigenous communities dataset: {ip_master.shape[0]} records')
print(f'\nEthnic groups identified:')
print(ip_master['ethnic'].value_counts())

Master indigenous communities dataset: 58 records

Ethnic groups identified:
ethnic
Jarai      17
Kreung     15
Tumpoun    15
Brao        8
Kachak      1
Kalaet      1
Kavet       1
Name: count, dtype: int64


In [19]:
# Registered communal lands with family/household data
ip_registered = ip_reg_filtered[[
    'map_id', 'ip_name', 'num_family', 'village', 'commune', 'district', 'province',
    'land_size', 'titled_par', 'residentia', 'tradi_agri', 'geometry'
]].copy()

ip_registered['num_family'] = pd.to_numeric(ip_registered['num_family'], errors='coerce')
ip_registered['land_size'] = pd.to_numeric(ip_registered['land_size'], errors='coerce')
ip_registered['x'] = ip_registered.geometry.apply(lambda geom: geom.geoms[0].x if hasattr(geom, 'geoms') else geom.x)
ip_registered['y'] = ip_registered.geometry.apply(lambda geom: geom.geoms[0].y if hasattr(geom, 'geoms') else geom.y)

print(f'Registered indigenous lands: {ip_registered.shape[0]} properties')
print(f'\nFamily statistics:')
print(f'  Total families: {ip_registered["num_family"].sum():.0f}')
print(f'  Avg families per property: {ip_registered["num_family"].mean():.1f}')
print(f'  Total land area: {ip_registered["land_size"].sum():.2f} sq km')

Registered indigenous lands: 44 properties

Family statistics:
  Total families: 4234
  Avg families per property: 111.4
  Total land area: 24939.00 sq km


In [20]:
# Try to load population boundaries (compressed format)
import gzip
import shutil

gpkg_path = DATA_DIR / 'kontur_boundaries_KH_20230628.gpkg.gz'
gpkg_uncompressed = DATA_DIR / 'kontur_boundaries_KH_20230628.gpkg'

try:
    if gpkg_path.exists() and not gpkg_uncompressed.exists():
        print('Decompressing boundaries file...')
        with gzip.open(gpkg_path, 'rb') as f_in:
            with open(gpkg_uncompressed, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
    
    if gpkg_uncompressed.exists():
        boundaries = gpd.read_file(gpkg_uncompressed)
        print(f'Population boundaries shape: {boundaries.shape}')
        print(f'Available columns: {boundaries.columns.tolist()}')
    else:
        print('Boundaries file not found')
except Exception as e:
    print(f'Note: Population boundaries processing: {str(e)[:100]}')

Decompressing boundaries file...
Note: Population boundaries processing: module 'fiona' has no attribute 'path'


## 7. Synthesis: Create Analysis-Ready Datasets

In [21]:
# Provincial-level summary dataset
provincial_summary = pd.DataFrame({
    'province': TARGET_PROVINCES
})

# Add indigenous community counts
mrd_counts = ip_mrd_filtered.groupby('province').size().reset_index(name='mrd_villages')
reg_counts = ip_reg_filtered.groupby('province').agg({
    'map_id': 'size',
    'num_family': 'sum'
}).reset_index().rename(columns={
    'map_id': 'registered_areas',
    'num_family': 'total_families'
})

provincial_summary = provincial_summary.merge(mrd_counts, on='province', how='left')
provincial_summary = provincial_summary.merge(reg_counts, on='province', how='left')
provincial_summary = provincial_summary.merge(flood_risk, on='province', how='left')

print('Provincial Summary (NE Cambodia):')
print(provincial_summary.to_string())

Provincial Summary (NE Cambodia):
       province  mrd_villages  registered_areas                                                                                        total_families  total_pop_exposed  total_area_flooded_km2  affected_districts  risk_score
0    Ratanakiri          58.0              27.0  Not foundNot found20596165737043106129576410617050Not foundNot foundNot found57531597210019734692156                NaN                     NaN                 NaN         NaN
1    Mondulkiri           NaN               8.0                                                                             1162745319380Not found105                NaN                     NaN                 NaN         NaN
2        Kratie           NaN               6.0                                                                                     37132223237140126          1095896.0                13281.36                 6.0   14377.256
3   Stung Treng           NaN               3.0                   

In [22]:
# Village-level dataset combining MRD communities with flood/risk context
villages_geospatial = ip_master.copy()

# Merge with flood risk by province
villages_geospatial = villages_geospatial.merge(
    flood_risk[['province', 'total_pop_exposed', 'total_area_flooded_km2', 'risk_score']],
    on='province',
    how='left'
)

# Add registered land context
reg_summary = ip_reg_filtered.groupby('province').agg({
    'num_family': 'sum',
    'land_size': 'sum'
}).reset_index().rename(columns={
    'num_family': 'total_families_region',
    'land_size': 'total_land_area_km2'
})

villages_geospatial = villages_geospatial.merge(reg_summary, on='province', how='left')

print(f'Village-level geospatial dataset: {villages_geospatial.shape[0]} records')
villages_geospatial.head()

Village-level geospatial dataset: 58 records


,map_id,ethnic,village,commune,district,province,requested_date_by_community,identified_by_mrd,geometry,data_source,x,y,total_pop_exposed,total_area_flooded_km2,risk_score,total_families_region,total_land_area_km2
0,1,Jarai,Le,Lum Choar,Ou Ya Dav,Ratanakiri,05/06/2010,Yes,MULTIPOINT (753135.511 1516543.261),MRD_communities,753135.511303,1.516543e+06,NaN,NaN,NaN,Not foundNot found2059616573704310612957641061...,588.48159201496.3127605.8134846.8997744.914851...
1,2,Kreung,Kralong,L'ak,Ou Chum,Ratanakiri,11/03/2011,Yes,MULTIPOINT (724489.697 1532509.079),MRD_communities,724489.697201,1.532509e+06,NaN,NaN,NaN,Not foundNot found2059616573704310612957641061...,588.48159201496.3127605.8134846.8997744.914851...
2,3,Jarai,Sala,Kak,Bar Kaev,Ratanakiri,03/04/2010,Yes,MULTIPOINT (741067.605 1515584.243),MRD_communities,741067.605297,1.515584e+06,NaN,NaN,NaN,Not foundNot found2059616573704310612957641061...,588.48159201496.3127605.8134846.8997744.914851...
3,4,Jarai,Ta Kok Chor Ray,Bar Kham,Ou Ya Dav,Ratanakiri,12/06/2011,Yes,MULTIPOINT (747949.578 1503741.348),MRD_communities,747949.577909,1.503741e+06,NaN,NaN,NaN,Not foundNot found2059616573704310612957641061...,588.48159201496.3127605.8134846.8997744.914851...
4,5,Kreung,Teum,Teun,Koun Mom,Ratanakiri,04/29/2013,Yes,MULTIPOINT (700969.915 1512923.183),MRD_communities,700969.914736,1.512923e+06,NaN,NaN,NaN,Not foundNot found2059616573704310612957641061...,588.48159201496.3127605.8134846.8997744.914851...


## 8. Load: Export Final Datasets

In [23]:
# Export to CSV (non-geographic summaries)
output_dir = Path('./processed_data')
output_dir.mkdir(exist_ok=True)

# Provincial summary
provincial_summary.to_csv(output_dir / '01_provincial_summary.csv', index=False)
print('✓ Exported: provincial_summary.csv')

# Flood risk
flood_risk.to_csv(output_dir / '02_flood_risk_analysis.csv', index=False)
print('✓ Exported: flood_risk_analysis.csv')

# Findex connectivity summary
findex_summary = findex_connectivity.describe().transpose()
findex_summary.to_csv(output_dir / '03_connectivity_statistics.csv')
print('✓ Exported: connectivity_statistics.csv')

✓ Exported: provincial_summary.csv
✓ Exported: flood_risk_analysis.csv
✓ Exported: connectivity_statistics.csv


In [24]:
# Export to GeoJSON (geospatial data)

# Indigenous communities
ip_master.to_file(
    output_dir / '04_indigenous_villages_mrd.geojson',
    driver='GeoJSON'
)
print('✓ Exported: indigenous_villages_mrd.geojson')

# Registered communal lands
ip_reg_filtered.to_file(
    output_dir / '05_indigenous_registered_lands.geojson',
    driver='GeoJSON'
)
print('✓ Exported: indigenous_registered_lands.geojson')

# Village-level geospatial
villages_geospatial.to_file(
    output_dir / '06_villages_with_risk_context.geojson',
    driver='GeoJSON'
)
print('✓ Exported: villages_with_risk_context.geojson')

✓ Exported: indigenous_villages_mrd.geojson
✓ Exported: indigenous_registered_lands.geojson
✓ Exported: villages_with_risk_context.geojson
